# SQL Murder Mystery

## Can you find out whodunnit?

![A decorative illustration of a detective looking at an evidence board.](https://mystery.knightlab.com/174092-clue-illustration.png)

There's been a Murder in SQL City! The SQL Murder Mystery is designed to be both a self-directed lesson to learn SQL concepts and commands and a fun game for experienced SQL users to solve an intriguing crime.

## SQL sleuths start here

A crime has taken place and the detective needs your help. The detective gave you the crime scene report, but you somehow lost it. You vaguely remember that the crime was a **​murder​**that occurred sometime on ​**Jan.15, 2018​** and that it took place in ​**SQL City​**. Start by retrieving the corresponding crime scene report from the police department’s database.

### Exploring the Database Structure

Experienced SQL users can often use database queries to infer the structure of a database. But each database system has different ways of managing this information. The SQL Murder Mystery is built using SQLite. Use this SQL command to find the tables in the Murder Mystery database.

Run this query to find the names of the tables in this database.

This command is specific to SQLite. For other databases, you'll have to learn their specific syntax.


In [2]:
# Install required packages
%pip install jupysql sqlalchemy pandas --quiet

# Load SQL magic
%load_ext sql

# Connect to the database
%sql sqlite:///sql-murder-mystery.db
#%config SqlMagic.style = 'table'


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Connecting to 'sqlite:///sql-murder-mystery.db'

In [3]:
%%sql
SELECT name
FROM sqlite_master

Running query in 'sqlite:///sql-murder-mystery.db'

name
crime_scene_report
drivers_license
facebook_event_checkin
interview
get_fit_now_member
sqlite_autoindex_get_fit_now_member_1
get_fit_now_check_in
solution
check_solution
income



Besides knowing the table names, you need to know how each table is structured. The way this works is also dependent upon which database technology you use. Here's how you do it with SQLite.

Run this query to find the structure of the `crime_scene_report` table

Change the value of 'name' to see the structure of the other tables you learned about with the previous query.


In [4]:
%%sql
SELECT sql
FROM sqlite_master
where name = 'crime_scene_report'

Running query in 'sqlite:///sql-murder-mystery.db'

sql
"CREATE TABLE crime_scene_report ( date integer, type text, description text, city text )"



### The rest is up to you!

If you're really comfortable with SQL, you can probably get it from here. To help, here is the schema diagram:

![schema diagram](schema.png)

Use your knowledge of the database schema and SQL commands to find out who committed the murder.
### Check your solution

Did you find the killer? When you think you know the answer, submit your suspect using the following code and find out if you're right.


In [95]:
%%sql
INSERT INTO solution VALUES (1, 'Miranda Priestly');
SELECT value FROM solution;

Running query in 'sqlite:///sql-murder-mystery.db'

1 rows affected.

value
"Congrats, you found the brains behind the murder! Everyone in SQL City hails you as the greatest SQL detective of all time. Time to break out the champagne!"


In [ ]:
%%sql
SELECT * FROM interview
WHERE person_id = '67318';

Running query in 'sqlite:///sql-murder-mystery.db'

person_id,transcript
67318,"I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5"" (65"") or 5'7"" (67""). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017."


In [86]:
%%sql
SELECT 
    person.id, 
    person.name, 
    person.license_id,
    COUNT(*) AS checkin_count
FROM person
JOIN facebook_event_checkin ON person.id = facebook_event_checkin.person_id
WHERE facebook_event_checkin.event_name = 'SQL Symphony Concert'
    AND facebook_event_checkin.date LIKE '201712%'
GROUP BY person.id, person.name, person.license_id
HAVING checkin_count = 3; 

Running query in 'sqlite:///sql-murder-mystery.db'

id,name,license_id,checkin_count
24556,Bryan Pardo,101191,3
99716,Miranda Priestly,202298,3


In [94]:
%%sql
SELECT 
    drivers_license.height,
    drivers_license.gender, 
    drivers_license.car_make, 
    drivers_license.hair_color,
    drivers_license.id,
    person.id, 
    person.name, 
    person.license_id
FROM person
JOIN drivers_license ON person.license_id = drivers_license.id
WHERE drivers_license.gender = 'female'
    AND drivers_license.car_make = 'Tesla'
    AND drivers_license.hair_color = 'red'
    AND drivers_license.id = '202298'

Running query in 'sqlite:///sql-murder-mystery.db'

height,gender,car_make,hair_color,id,id_1,name,license_id
66,female,Tesla,red,202298,99716,Miranda Priestly,202298


FINAL MURDERER IS MIRANDA PRIESTLY!

How I arrrived to that conclusion:

I ran a query to understand how crime_scene_report table is structured, then filtered the crime scene report that occured on January 15, 2018 in "SQL City"

In [6]:
%%sql
SELECT * FROM crime_scene_report LIMIT 5;

Running query in 'sqlite:///sql-murder-mystery.db'

date,type,description,city
20180115,robbery,A Man Dressed as Spider-Man Is on a Robbery Spree,NYC
20180115,murder,Life? Dont talk to me about life.,Albany
20180115,murder,"Mama, I killed a man, put a gun against his head...",Reno
20180215,murder,REDACTED REDACTED REDACTED,SQL City
20180215,murder,Someone killed the guard! He took an arrow to the knee!,SQL City


In [ ]:
%%sql
SELECT * FROM crime_scene_report
WHERE date = '20180115' AND city = 'SQL City';

Running query in 'sqlite:///sql-murder-mystery.db'

date,type,description,city
20180115,assault,"Hamilton: Lee, do you yield? Burr: You shot him in the side! Yes he yields!",SQL City
20180115,assault,Report Not Found,SQL City
20180115,murder,"Security footage shows that there were 2 witnesses. The first witness lives at the last house on ""Northwestern Dr"". The second witness, named Annabel, lives somewhere on ""Franklin Ave"".",SQL City


Further investigating the murder that occured on January 15, 208 in SQL City. 
1. The first witness lives at the last house on "Northwestern Dr". 
2. The second witness, named Annabel, lives somewhere on "Franklin Ave".

In [8]:
%%sql
SELECT * FROM person LIMIT 5;

Running query in 'sqlite:///sql-murder-mystery.db'

id,name,license_id,address_number,address_street_name,ssn
10000,Christoper Peteuil,993845,624,Bankhall Ave,747714076
10007,Kourtney Calderwood,861794,2791,Gustavus Blvd,477972044
10010,Muoi Cary,385336,741,Northwestern Dr,828638512
10016,Era Moselle,431897,1987,Wood Glade St,614621061
10025,Trena Hornby,550890,276,Daws Hill Way,223877684


In [9]:
%%sql
SELECT * FROM person
WHERE address_street_name = ('Northwestern Dr') 

Running query in 'sqlite:///sql-murder-mystery.db'

id,name,license_id,address_number,address_street_name,ssn
10010,Muoi Cary,385336,741,Northwestern Dr,828638512
12711,Norman Apolito,667757,599,Northwestern Dr,778264744
14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949
15171,Weldon Penso,336999,311,Northwestern Dr,131379495
17729,Lasonya Wildey,439686,3824,Northwestern Dr,917817122
18376,Josh Shi,653712,1091,Northwestern Dr,193899001
19420,Cody Schiel,890431,3524,Northwestern Dr,947110049
22239,Dusty Sigafus,710517,1125,Northwestern Dr,724386723
23044,Val Portlock,924989,3143,Northwestern Dr,100593316
23960,Kristopher Lagerberg,658777,1392,Northwestern Dr,492912529


In [10]:
%%sql
SELECT * FROM person
WHERE address_street_name = ('Franklin Ave') 
    AND name LIKE ('Annabel %');

Running query in 'sqlite:///sql-murder-mystery.db'

id,name,license_id,address_number,address_street_name,ssn
16371,Annabel Miller,490173,103,Franklin Ave,318771143


Two witnesses are:

| #  | id    | name            | license_id | address_number | address_street_name | ssn        |
|----|-------|------------------|-------------|----------------|----------------------|-------------|
| 1. | 14887 | Morty Schapiro   | 118009      | 4919           | Northwestern Dr       | 111564949   |
| 2. | 16371 | Annabel Miller   | 490173      | 103            | Franklin Ave          | 318771143   |

In [11]:
%%sql
SELECT * FROM interview LIMIT 5;

Running query in 'sqlite:///sql-murder-mystery.db'

person_id,transcript
28508,‘I deny it!’ said the March Hare.
63713,
86208,"way, and the whole party swam to the shore."
35267,"lessons in here? Why, there’s hardly room for YOU, and no room at all"
33856,


In [12]:
%%sql
SELECT * FROM interview
WHERE person_id IN ('14887', '16371');

Running query in 'sqlite:///sql-murder-mystery.db'

person_id,transcript
14887,"I heard a gunshot and then saw a man run out. He had a ""Get Fit Now Gym"" bag. The membership number on the bag started with ""48Z"". Only gold members have those bags. The man got into a car with a plate that included ""H42W""."
16371,"I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th."


Murderer details: 
- "Get Fit Now Gym" bag
- Male (drivers_license --> "gender")
- Car plate included "H42W" (drivers_license --> "plate_number")

- Membership number started with "48Z" (get_fit_now_check_in --> "membership_id")
- January 9th (get_fit_now_check_in --> "check_in_date" )

- Gold member (get_fit_now_member --> "membership_status")

In [14]:
%%sql
SELECT * FROM get_fit_now_check_in
WHERE membership_id LIKE ('48Z%')
    AND check_in_date LIKE ('%0109')

Running query in 'sqlite:///sql-murder-mystery.db'

membership_id,check_in_date,check_in_time,check_out_time
48Z7A,20180109,1600,1730
48Z55,20180109,1530,1700


Only two members who checked in on 01/09/2018: 

membership_id: 
1. 48Z7A
2. 48Z55

In [15]:
%%sql
SELECT * FROM get_fit_now_member
WHERE membership_status = ('gold')
    AND id IN ('48Z7A', '48Z55');

Running query in 'sqlite:///sql-murder-mystery.db'

id,person_id,name,membership_start_date,membership_status
48Z55,67318,Jeremy Bowers,20160101,gold
48Z7A,28819,Joe Germuska,20160305,gold


Suspects: 

1. Jeremy Bowers (person_id: 67318, id: 48Z55)
2. Joe Germuska (person_id: 28819, id: 48Z7A)

In [16]:
%%sql
SELECT * FROM drivers_license
WHERE gender = ('male')
    AND plate_number LIKE ('%H42W%')

Running query in 'sqlite:///sql-murder-mystery.db'

id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
423327,30,70,brown,brown,male,0H42W2,Chevrolet,Spark LS
664760,21,71,black,black,male,4H42WR,Nissan,Altima


I'm using the "id" from "drivers_license" and checking that with "id" from "person" table in order to see if one of the suspects matches all of the clues I have. 

In [37]:
%%sql
SELECT 
    person.id,
    person.name,
    person.license_id,
    drivers_license.id,
    drivers_license.gender,
    drivers_license.plate_number
FROM person
JOIN drivers_license
    ON person.license_id = drivers_license.id
WHERE person.id IN ('67318', '28819')


Running query in 'sqlite:///sql-murder-mystery.db'

id,name,license_id,id_1,gender,plate_number
67318,Jeremy Bowers,423327,423327,male,0H42W2


Just to double check, I'm running Joe Germuska's "id" (28819) in the "person" table to check his "license_id", then using that to check his "plate_number" in "drivers_license" table. 

In [53]:
%%sql
SELECT * FROM person
WHERE id = '28819';  -- Joe Germuska's license_id is 173289


SELECT * FROM drivers_license
WHERE id = '173289';


Running query in 'sqlite:///sql-murder-mystery.db'

id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model


Final suspect is Jeremy Bowers (person_id: 67318) since he checks against all of the clues obtained. 